# Imports

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import sqlite3
from datetime import date

import pandas as pd
from openai import OpenAI


# Convert csv to SQLite

In [ ]:
llm_client = OpenAI()

CSV_FILE = "data/transactions.csv"
DB_FILE = "data/transactions.db"

In [ ]:
def csv_to_sqlite(csv_file, db_file):

    df = pd.read_csv(csv_file)

    connection = sqlite3.connect(db_file)

    df.to_sql(
        "transactions",
        connection,
        if_exists="replace",
        index=False
    )

    connection.close()

    print(f"Loaded {len(df)} rows into SQLite.")


In [ ]:
csv_to_sqlite(CSV_FILE,DB_FILE)

# SQL RAG

## extract schema

In [13]:
def get_schema(db_file):

    connection = sqlite3.connect(db_file)

    cursor = connection.cursor()

    cursor.execute("""
        SELECT
            name,
            type
        FROM pragma_table_info('transactions')
    """)

    columns = cursor.fetchall()

    connection.close()

    schema = "Table: transactions\n\n"

    for column_name, column_type in columns:
        schema += f"- {column_name}: {column_type}\n"

    return schema

In [14]:
print(get_schema(DB_FILE))

Table: transactions

- transaction_id: TEXT
- date: TEXT
- store: TEXT
- region: TEXT
- product: TEXT
- quantity: INTEGER
- price: INTEGER
- revenue: INTEGER



## Generate SQL from user question

In [ ]:
def generate_sql(user_query, schema):

    today = date.today().isoformat()

    prompt = f"""
You are a SQL expert.

Convert the user's natural-language question into a
SQLite SQL query.

Today's date is: {today}

Database schema:

{schema}

Rules:

1. Use only the table and columns provided in the schema.
2. Return SQL only.
3. Do not use Markdown code fences.
4. Do not modify, insert, update, or delete data.
5. Only generate SELECT queries.
6. Use SQLite-compatible SQL.
7. When the user says "last quarter", calculate the
   previous calendar quarter based on today's date.
8. Use explicit date boundaries when working with quarters.

User question:

{user_query}
"""

    response = llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You generate safe SQLite SELECT queries."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    sql = response.choices[0].message.content.strip()

    # Remove accidental markdown fences
    if sql.startswith("```"):
        sql = sql.replace("```sql", "")
        sql = sql.replace("```", "")
        sql = sql.strip()

    return sql


## Execute SQL


In [ ]:
def execute_sql(sql, db_file):

    connection = sqlite3.connect(db_file)

    try:

        # Basic safety check
        sql_upper = sql.strip().upper()

        if not sql_upper.startswith("SELECT"):
            raise ValueError("Only SELECT queries are allowed.")

        result = pd.read_sql_query(
            sql,
            connection
        )

        return result

    finally:
        connection.close()

## Generate final answer

In [ ]:
def generate_answer(user_query, sql, sql_result):

    prompt = f"""
Answer the user's question using the SQL result.

User question:
{user_query}

SQL query:
{sql}

SQL result:
{sql_result.to_string(index=False)}

Instructions:

- Answer directly.
- Do not invent information.
- Use the values from the SQL result.
- Keep the answer concise.
"""

    response = llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You answer questions using database results."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

## Complete SQL RAG pipeline

In [15]:
def sql_rag(user_query):

    schema = get_schema(DB_FILE)

    # LLM → SQL
    sql = generate_sql(
        user_query,
        schema
    )

    print("\nGenerated SQL:")
    print(sql)
    print("*"*20)
    # SQL → Database
    sql_result = execute_sql(
        sql,
        DB_FILE
    )

    print("\nSQL Result:")
    print(sql_result)

    # Database result → LLM
    answer = generate_answer(
        user_query,
        sql,
        sql_result
    )

    return answer

In [16]:
user_query = "What was total revenue by region last quarter?"

answer = sql_rag(user_query)

print("\nFinal Answer:")
print(answer)


Generated SQL:
SELECT region, SUM(revenue) AS total_revenue
FROM transactions
WHERE date >= '2026-04-01' AND date < '2026-07-01'
GROUP BY region;
********************

SQL Result:
  region  total_revenue
0   East         338000
1  North         303000
2  South         350000
3   West         358000

Final Answer:
Total revenue by region last quarter was as follows:
- East: $338,000
- North: $303,000
- South: $350,000
- West: $358,000
